# 04 — Nested walk-forward HPO (Case B)

Tune `HistGradientBoostingRegressor` hyperparameters on a **validation** expanding-window,
then evaluate the chosen config once on a held-out **test** window.

| Window | Score weeks | Role |
|---|---|---|
| Val | `2025-01-07` → `2025-06-24` | Choose θ* |
| Test | `≥ 2025-07-01` | Final report (untouched during tuning) |

Train at each Tuesday `t` remains expanding (`date < t`). Features: diesel **level**
(`diesel_us` + `diesel_distance = diesel_us × distance/1000`).

**Selection rule:** among configs with cold-start (`0-4`) MAE lift ≥ 0 vs lag-1 on val,
pick the lowest overall MAE; if none qualify, pick best cold lift then lowest MAE.

Writes `models/nested_hpo/best_params.json`. Next step: notebook **05** for full-window
results and visualizations with θ*.


In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path

import pandas as pd

from freight_rates.evaluation import dual_regime_headline, format_dual_regime_report
from freight_rates.ingestion import load_raw_snapshot
from freight_rates.preprocessing import build_modeling_panel
from freight_rates.splits import (
    DIAGNOSTIC_TEST_START,
    DIAGNOSTIC_VAL_END,
    DIAGNOSTIC_VAL_START,
    FIRST_FORECAST_DATE,
    filter_model_window,
)
from freight_rates.walkforward import default_gbm, run_walkforward_gbm

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "models" / "nested_hpo"
OUT.mkdir(parents=True, exist_ok=True)

VAL_FIRST = DIAGNOSTIC_VAL_START  # 2025-01-07
VAL_LAST = DIAGNOSTIC_VAL_END  # 2025-06-24
TEST_FIRST = DIAGNOSTIC_TEST_START  # 2025-07-01

print(f"FIRST_FORECAST_DATE = {FIRST_FORECAST_DATE.date()}")
print(f"val window:  {VAL_FIRST.date()} → {VAL_LAST.date()}")
print(f"test window: ≥ {TEST_FIRST.date()}")


FIRST_FORECAST_DATE = 2025-01-07
val window:  2025-01-07 → 2025-06-24
test window: ≥ 2025-07-01


In [2]:
panel = filter_model_window(build_modeling_panel(load_raw_snapshot(raw_dir=RAW), raw_dir=RAW))
print(
    f"panel: {len(panel):,} rows  "
    f"{panel['date'].min().date()} → {panel['date'].max().date()}"
)
assert "diesel_us" in panel.columns


panel: 8,202 rows  2024-07-02 → 2026-08-18


## Hyperparameter grid

Modest grid focused on regularization (madure lanes overfit lag-1 with deep trees).


In [3]:
PARAM_GRID = {
    "max_depth": [3, 4, 6],
    "min_samples_leaf": [20, 40, 60],
    "l2_regularization": [0.1, 1.0],
    "learning_rate": [0.08],  # fixed; keep grid tractable
    "max_iter": [200],
}

keys = list(PARAM_GRID.keys())
configs = [dict(zip(keys, vals)) for vals in product(*(PARAM_GRID[k] for k in keys))]
print(f"n configs = {len(configs)}")
pd.DataFrame(configs)


n configs = 18


,max_depth,min_samples_leaf,l2_regularization,learning_rate,max_iter
0,3,20,0.1,0.08,200
1,3,20,1.0,0.08,200
2,3,40,0.1,0.08,200
3,3,40,1.0,0.08,200
4,3,60,0.1,0.08,200
5,3,60,1.0,0.08,200
6,4,20,0.1,0.08,200
7,4,20,1.0,0.08,200
8,4,40,0.1,0.08,200
9,4,40,1.0,0.08,200


## Val walk-forward — tune θ*

For each config, score expanding-window folds with
`first_forecast_date = val start`, `last_forecast_date = val end`.


In [4]:
def score_config(params: dict, *, first, last, label: str) -> tuple[pd.DataFrame, object]:
    """Run walk-forward for one θ; return dual-regime headline + full result."""
    result = run_walkforward_gbm(
        panel,
        first_forecast_date=first,
        last_forecast_date=last,
        residual_target=True,
        diesel_features="level",
        model=default_gbm(**params),
        verbose=False,
    )
    headline = dual_regime_headline(result.overall, result.by_history, label=label)
    for k, v in params.items():
        headline[k] = v
    return headline, result


val_rows = []
val_by_week_store: dict[str, pd.DataFrame] = {}

for i, params in enumerate(configs):
    label = f"cfg_{i:02d}"
    print(f"[{i + 1}/{len(configs)}] val {label} {params}")
    headline, result = score_config(params, first=VAL_FIRST, last=VAL_LAST, label=label)
    val_rows.append(headline)
    val_by_week_store[label] = result.by_week.assign(config=label, **params)

val_summary = pd.concat(val_rows, ignore_index=True)
val_summary_path = OUT / "hpo_val_summary.csv"
val_summary.to_csv(val_summary_path, index=False)
print("wrote", val_summary_path)
display(
    val_summary[
        [
            "label",
            "max_depth",
            "min_samples_leaf",
            "l2_regularization",
            "mae",
            "mae_lift",
            "cold_mae_lift",
            "beats_lag1_cold",
            "n",
            "cold_n",
        ]
    ].sort_values(["beats_lag1_cold", "mae"], ascending=[False, True])
)


[1/18] val cfg_00 {'max_depth': 3, 'min_samples_leaf': 20, 'l2_regularization': 0.1, 'learning_rate': 0.08, 'max_iter': 200}
[2/18] val cfg_01 {'max_depth': 3, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'learning_rate': 0.08, 'max_iter': 200}
[3/18] val cfg_02 {'max_depth': 3, 'min_samples_leaf': 40, 'l2_regularization': 0.1, 'learning_rate': 0.08, 'max_iter': 200}
[4/18] val cfg_03 {'max_depth': 3, 'min_samples_leaf': 40, 'l2_regularization': 1.0, 'learning_rate': 0.08, 'max_iter': 200}
[5/18] val cfg_04 {'max_depth': 3, 'min_samples_leaf': 60, 'l2_regularization': 0.1, 'learning_rate': 0.08, 'max_iter': 200}
[6/18] val cfg_05 {'max_depth': 3, 'min_samples_leaf': 60, 'l2_regularization': 1.0, 'learning_rate': 0.08, 'max_iter': 200}
[7/18] val cfg_06 {'max_depth': 4, 'min_samples_leaf': 20, 'l2_regularization': 0.1, 'learning_rate': 0.08, 'max_iter': 200}
[8/18] val cfg_07 {'max_depth': 4, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'learning_rate': 0.08, 'max_iter': 200}


KeyboardInterrupt: 

In [ ]:
# Selection: prefer cold lift ≥ 0, then lowest overall MAE on val.
eligible = val_summary.loc[val_summary["cold_mae_lift"] >= 0].copy()
if eligible.empty:
    print("No config with cold_mae_lift ≥ 0; falling back to best cold lift then MAE.")
    best_row = val_summary.sort_values(
        ["cold_mae_lift", "mae"], ascending=[False, True]
    ).iloc[0]
else:
    best_row = eligible.sort_values("mae", ascending=True).iloc[0]

best_params = {
    "max_depth": int(best_row["max_depth"]),
    "min_samples_leaf": int(best_row["min_samples_leaf"]),
    "l2_regularization": float(best_row["l2_regularization"]),
    "learning_rate": float(best_row["learning_rate"]),
    "max_iter": int(best_row["max_iter"]),
}
best_label = str(best_row["label"])
print("θ* =", best_params)
print(format_dual_regime_report(best_row.to_frame().T))

pd.Series({"config": best_label, **best_params}).to_json(OUT / "best_params.json")


### Val folds — dates and metrics for θ*


In [ ]:
val_folds = val_by_week_store[best_label].copy()
val_folds["date"] = pd.to_datetime(val_folds["date"])
val_folds = val_folds.sort_values("date").reset_index(drop=True)
val_folds.insert(0, "fold", range(1, len(val_folds) + 1))

val_fold_cols = [
    "fold",
    "date",
    "n",
    "mae",
    "mae_baseline",
    "mae_lift",
    "medae",
    "mape",
]
val_fold_table = val_folds[val_fold_cols]
val_fold_table.to_csv(OUT / "hpo_val_folds_best.csv", index=False)

print(
    f"Val folds for {best_label}: n_folds={len(val_fold_table)}, "
    f"{val_fold_table['date'].min().date()} → {val_fold_table['date'].max().date()}"
)
display(val_fold_table)
print("Val aggregate:")
display(
    val_fold_table[["n", "mae", "mae_baseline", "mae_lift"]]
    .agg({"n": "sum", "mae": "mean", "mae_baseline": "mean", "mae_lift": "mean"})
    .to_frame()
    .T
)


## Test holdout — evaluate θ* once

Score weeks `≥ 2025-07-01`. Expanding train includes all history before each `t`
(including the val period) — standard nested protocol.


In [ ]:
print("Running test holdout with θ*…")
test_headline, test_result = score_config(
    best_params,
    first=TEST_FIRST,
    last=None,
    label="test_best",
)
print(format_dual_regime_report(test_headline))

test_folds = test_result.by_week.copy()
test_folds["date"] = pd.to_datetime(test_folds["date"])
test_folds = test_folds.sort_values("date").reset_index(drop=True)
test_folds.insert(0, "fold", range(1, len(test_folds) + 1))
test_fold_table = test_folds[
    ["fold", "date", "n", "mae", "mae_baseline", "mae_lift", "medae", "mape"]
]
test_fold_table.to_csv(OUT / "hpo_test_folds_best.csv", index=False)
test_headline.to_csv(OUT / "hpo_test_dual_regime.csv", index=False)

print(
    f"Test folds: n_folds={len(test_fold_table)}, "
    f"{test_fold_table['date'].min().date()} → {test_fold_table['date'].max().date()}"
)
display(test_fold_table)
display(test_headline)


## Side-by-side: val θ* vs test θ*


In [ ]:
compare = pd.concat(
    [
        best_row.to_frame().T.assign(window="val"),
        test_headline.assign(window="test"),
    ],
    ignore_index=True,
)
cols = [
    "window",
    "label",
    "n",
    "mae",
    "mae_baseline",
    "mae_lift",
    "cold_n",
    "cold_mae",
    "cold_mae_lift",
    "beats_lag1_overall",
    "beats_lag1_cold",
]
display(compare[cols])
compare[cols].to_csv(OUT / "hpo_val_vs_test.csv", index=False)

print("Artifacts under", OUT)
for p in sorted(OUT.glob("*")):
    print(" -", p.name)
